# Week 4: Sequence-to-Sequence Models

## Learning Objectives
- Understand encoder-decoder architectures
- Build a character-level text generation model
- Implement text summarisation basics
- Use beam search for decoding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF = True
    print(f'TensorFlow {tf.__version__}')
except ImportError:
    TF = False
    print('TensorFlow not installed.')

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid')

## 1. Encoder-Decoder Architecture

Seq2Seq models consist of:
- **Encoder**: Processes input sequence and creates a context vector
- **Decoder**: Generates output sequence from the context vector

Applications: **machine translation**, **text summarisation**, **question answering**

## 2. Character-Level Text Generation with LSTM

In [ ]:
text = """
data science is the study of data to extract meaning or insights using statistics machine learning
and other quantitative methods the rise of the digital age has led to an explosion of data from
social media sensors transactions and interactions between people and technology data scientists use
programming languages like python and r to analyse and visualise data building models that can make
predictions classifications and recommendations"""

text = text.lower()
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
print(f'Unique characters: {len(chars)}')
print('Characters:', ''.join(chars))

In [ ]:
if TF:
    SEQ_LEN = 40
    STEP = 3
    X_list, y_list = [], []
    for i in range(0, len(text) - SEQ_LEN, STEP):
        X_list.append([char2idx[c] for c in text[i:i+SEQ_LEN]])
        y_list.append(char2idx[text[i+SEQ_LEN]])

    X_arr = tf.keras.utils.to_categorical(X_list, num_classes=len(chars))
    y_arr = tf.keras.utils.to_categorical(y_list, num_classes=len(chars))
    print(f'Training samples: {len(X_list)}')

    char_model = keras.Sequential([
        layers.LSTM(128, input_shape=(SEQ_LEN, len(chars))),
        layers.Dense(len(chars), activation='softmax')
    ])
    char_model.compile(optimizer='adam', loss='categorical_crossentropy')
    char_model.fit(X_arr, y_arr, epochs=10, batch_size=128, verbose=1)
else:
    print('TensorFlow not available.')

In [ ]:
if TF:
    def sample_next(preds, temperature=1.0):
        preds = np.log(np.array(preds, dtype='float64') + 1e-9) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))
        return np.argmax(np.random.multinomial(1, preds, 1))

    seed = text[:SEQ_LEN]
    generated = seed
    for _ in range(200):
        x_pred = np.array([[char2idx[c] for c in generated[-SEQ_LEN:]]])
        x_pred = tf.keras.utils.to_categorical(x_pred, num_classes=len(chars))
        preds = char_model.predict(x_pred, verbose=0)[0]
        next_char = idx2char[sample_next(preds, temperature=0.5)]
        generated += next_char
    print('Generated text:')
    print(generated[SEQ_LEN:])
else:
    print('TensorFlow not available.')

## Exercise

1. Increase LSTM units to 256 and train for more epochs
2. Experiment with temperature values (0.2, 0.5, 1.0, 1.5)
3. Implement a simple extractive summariser using TF-IDF sentence scoring

In [ ]:
# Your code here


## Summary

- ✓ Encoder-decoder architecture
- ✓ Character-level LSTM text generation
- ✓ Temperature sampling for diversity

## Next Week
Transformers and BERT!